[◀ Appendix A: DB Connectivity](Appendix_A_database_connectivity.ipynb) | [Table of Contents](TOC.ipynb) | [Appendix C: Serialization ▶](Appendix_C_serialization.ipynb)

# Appendix B: Regular Expressions (The re Module)

This appendix covers text pattern matching in Python using the standard library's `re` module, regex engine characteristics, group extraction, greedy vs. non-greedy matches, and lookarounds.

Reference: [re module](https://docs.python.org/3/library/re.html)


## 1. Regex Compilation and Core Functions

### Under the Hood: The Backtracking Engine
Python's `re` module uses a backtracking NFA (Nondeterministic Finite Automaton) engine. 
To speed up execution, all functions (like `re.search`, `re.match`) parse regex strings and compile them into internal code structures. They cache these compiled regex patterns automatically (up to a default limit). However, compiling patterns explicitly with **`re.compile()`** is recommended for readability and performance inside loop structures.

### Key API Actions
- **`re.match(pattern, string)`**: Checks for a match **only at the beginning** of the string.
- **`re.search(pattern, string)`**: Searches the **entire string** for the first match.
- **`re.findall(pattern, string)`**: Extracts all matching substrings as a list of strings.
- **`re.finditer(pattern, string)`**: Returns an **iterator** yielding match objects. This is highly memory-efficient for large strings because it processes matches lazily.

### Repetitive Extraction: Parsing Key-Value Sequences
A common log parsing and text processing task is to extract repeating patterns (like `key=value` or `key:value` sequences) from space-separated lines or strings.
For repetitive extraction across a line, `re.findall()` and `re.finditer()` are the primary tools.

#### Pattern Design for Key-Value Extraction
To extract `key=value` pairs:
1. Define a pattern that captures the key and value separately: `r"(\w+)=(\S+)"`.
   - `\w+` captures alphanumeric keys.
   - `=` matches the literal separator.
   - `\S+` captures non-whitespace characters for the value.
2. If values can be quoted or contain spaces, design a pattern that handles optional quoting, e.g. `r"(\w+)=(?:\"([^\"]*)\"|(\S+))"`.

#### Using `re.findall`
If the pattern contains multiple capture groups, `re.findall()` returns a list of tuples representing the captured groups. This list can be passed directly to the `dict` constructor:
```python
import re

log_line = "user_id=123 status=success duration=150ms"
kv_pattern = re.compile(r"(\w+)=(\S+)")

# findall returns a list of tuples: [('user_id', '123'), ('status', 'success'), ...]
pairs = kv_pattern.findall(log_line)
payload_dict = dict(pairs)
print(payload_dict)
# Output: {'user_id': '123', 'status': 'success', 'duration': '150ms'}
```

#### Using `re.finditer` for Lazy and Position-Aware Parsing
For larger strings or when you need exact match span index positions, use `re.finditer()`. It yields match objects lazily, conserving memory:
```python
payload_dict = {}
for match in kv_pattern.finditer(log_line):
    key = match.group(1)
    val = match.group(2)
    payload_dict[key] = val
    print(f"Extracted '{key}' at string indices {match.span()}")
```


In [ ]:
import re

text = "Errors: 404 Not Found, 500 Server Error"
pattern = r"\d{3}"

# 1. Compile regex pattern
regex = re.compile(pattern)

# 2. Match vs Search
print("match():", regex.match(text))    # None, since string doesn't start with digits
print("search():", regex.search(text).group())  # '404', matches first occurrence

# 3. Finditer (Lazy iterator)
for match in regex.finditer(text):
    print(f"Found match '{match.group()}' at indices {match.span()}")


## 2. Capture Groups and Named Groups

- **Numbered Capture Groups**: Defined by enclosing parentheses `(pattern)`. They are extracted using `.group(1)`, `.group(2)`, etc.
- **Named Capture Groups**: Defined by `(?P<name>pattern)`. This makes regex strings significantly more self-documenting. They are extracted using `.group('name')` or as a dictionary via `.groupdict()`.


In [ ]:
email_text = "Contact: alice.smith@example.com"
email_regex = re.compile(r"(?P<user>[\w.]+)@(?P<domain>[\w.]+)")

match = email_regex.search(email_text)
if match:
    print("User group:", match.group('user'))
    print("Domain group:", match.group('domain'))
    print("Groupdict dictionary:", match.groupdict())


## 3. Greedy vs. Non-Greedy Matching

By default, regex quantifiers (`*`, `+`, `?`) are **greedy**: they consume as much text as possible. 
Appending a question mark (`?`) turns them into **non-greedy** (or lazy) quantifiers, matching the *minimum* number of characters required to satisfy the expression.


In [ ]:
html = "<div>First</div><div>Second</div>"

# Greedy pattern matches from first < to last >
greedy = re.search(r"<div>.*</div>", html)
print("Greedy Match:", greedy.group())

# Non-greedy pattern matches the shortest possible sub-string
lazy = re.search(r"<div>.*?</div>", html)
print("Lazy Match:", lazy.group())


## 4. Lookarounds

Lookarounds are **zero-width assertions** that verify a condition without consuming string characters:
- **Positive Lookahead `(?=...)`**: Assures that the pattern *is* followed immediately.
- **Negative Lookahead `(?!...)`**: Assures that the pattern *is not* followed.
- **Positive Lookbehind `(?<=...)`**: Assures that the pattern *is* preceded immediately (must be of fixed length).
- **Negative Lookbehind `(?<!...)`**: Assures that the pattern *is not* preceded immediately.


In [ ]:
# 1. Positive Lookbehind: Match price values preceded by a dollar sign
prices = "Costs: $100, €50, $20"
usd_regex = re.compile(r"(?<=\$)\d+")
print("USD prices found:", usd_regex.findall(prices))

# 2. Negative Lookahead: Match words NOT followed by 'ing'
words = "talking, walk, singing, sing"
root_regex = re.compile(r"\b\w+(?!ing)\b")
# Matches 'walk' and 'sing', but not 'talking' or 'singing'
print("Root words:", root_regex.findall(words))


## 5. Coding Challenges

Complete the following exercises to test your understanding.


### Challenge 1: Robust Log Parser

Implement the function `parse_log_line(log_line)` which parses unstructured log statements.
A log line follows this layout:
`[TIMESTAMP] LEVEL: MESSAGE` where MESSAGE contains space-separated `key=value` metadata payloads.

Specifically, the parser should:
1. Match the exact pattern: `[YYYY-MM-DD HH:MM:SS] LEVEL: MESSAGE`
   - Use **named capture groups** to extract `timestamp` and `level` (e.g. INFO, ERROR, WARNING).
2. Parse the space-separated key-value pairs at the end of the line (e.g. `user_id=123 status=success`) and compile them into a dictionary assigned to the `'payload'` key.
3. Return a dictionary containing:
   `{'timestamp': timestamp_str, 'level': level_str, 'payload': payload_dict}`
4. If the log line is malformed or doesn't match the required format, return `None` immediately.


In [ ]:
def parse_log_line(log_line: str) -> dict | None:
    """
    Parses a log line containing a timestamp, level, and metadata key-values.
    Returns a dictionary or None if the log line does not conform to the pattern.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Run this cell to verify your implementation
try:
    # Test case 1: Valid INFO log line
    log1 = "[2026-06-02 12:00:00] INFO: Request completed for user_id=42 status=200 duration=150ms"
    res1 = parse_log_line(log1)
    assert res1 is not None, "Should not be None"
    assert res1['timestamp'] == "2026-06-02 12:00:00", "Failed to extract timestamp"
    assert res1['level'] == "INFO", "Failed to extract level"
    assert res1['payload'] == {
        "user_id": "42",
        "status": "200",
        "duration": "150ms"
    }, f"Payload parsing mismatch, got: {res1['payload']}"

    # Test case 2: Valid ERROR log line
    log2 = "[2026-06-02 12:05:30] ERROR: DB connection failed error_code=500 retry=false"
    res2 = parse_log_line(log2)
    assert res2['level'] == "ERROR"
    assert res2['payload'] == {"error_code": "500", "retry": "false"}

    # Test case 3: Malformed formats
    assert parse_log_line("No brackets log line") is None
    assert parse_log_line("[2026-06-02] INFO: missing HH:MM:SS") is None
    assert parse_log_line("[2026-06-02 12:00:00] info: level should be capitalized") is None
    
    print("🎉 Challenge 1 Passed Successfully!")
except AssertionError as e:
    print(f"❌ Test Failed: {e}")


### Challenge 2: HTML Link Extractor

Implement the function `extract_links(html)` that parses HTML content and extracts all links.
- Each link is represented by an anchor tag: `<a href="URL">LINK TEXT</a>`.
- The function should return a list of tuples: `[(url_1, text_1), (url_2, text_2), ...]`.
- Ensure your pattern handles arbitrary spacing inside the tag (e.g. `<a  href="..." >`) and is **non-greedy** so it doesn't merge adjacent anchor tags.


In [ ]:
def extract_links(html: str) -> list[tuple[str, str]]:
    """
    Extracts all links from a string of HTML code.
    Returns a list of tuples containing (href, link_text).
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Run this cell to verify your implementation
try:
    html_content = (
        '<p>Please visit <a href="https://www.python.org">Python Portal</a> or check '
        '<a  href="/docs/index.html">local docs</a>.</p>'
    )
    
    links = extract_links(html_content)
    assert len(links) == 2, "Should extract exactly 2 links"
    assert links[0] == ("https://www.python.org", "Python Portal"), "First link mismatch"
    assert links[1] == ("/docs/index.html", "local docs"), "Second link mismatch"
    
    # Test empty or no links
    assert extract_links("<div>No links here</div>") == [], "Should return empty list for no links"
    
    print("🎉 Challenge 2 Passed Successfully!")
except AssertionError as e:
    print(f"❌ Test Failed: {e}")


[◀ Appendix A: DB Connectivity](Appendix_A_database_connectivity.ipynb) | [Table of Contents](TOC.ipynb) | [Appendix C: Serialization ▶](Appendix_C_serialization.ipynb)